In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
current_path = "/content/drive/MyDrive/dl-techniques-01"
os.chdir(current_path)
print(os.getcwd())

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/1Eg_RxQg6QCabZhzAJqjn6fTmJWVvCAJs/dl-techniques-01


In [2]:
import numpy as np
import matplotlib.pyplot as plt

import json
import keras
import tensorflow as tf

from tensorflow.keras.applications.inception_resnet_v2 import preprocess_input

from keras import layers
from tensorflow.keras import applications
from keras.models import Sequential, load_model
from tensorflow.keras.preprocessing import image_dataset_from_directory

In [3]:
path = "./datasets/flowers"

img_width = 224
img_height = 224
batch_size = 32

In [4]:
train_ds = image_dataset_from_directory(
    path,
    seed=123,
    validation_split=0.2,
    subset='training',
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = image_dataset_from_directory(
    path,
    seed=123,
    validation_split=0.2,
    subset='validation',
    image_size=(img_height, img_width),
    batch_size=batch_size
)

Found 11200 files belonging to 7 classes.
Using 8960 files for training.
Found 11200 files belonging to 7 classes.
Using 2240 files for validation.


In [5]:
classes = train_ds.class_names

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

In [6]:
base_model = tf.keras.applications.InceptionResNetV2(include_top=False, input_shape=(img_height, img_width, 3), weights='imagenet')
base_model.trainable = False

data_agumentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomZoom(0.1),
    layers.RandomRotation(0.2)
])

219055592/219055592 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


In [9]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    patience=10,
    restore_best_weights=True
)

In [6]:
model = tf.keras.Sequential([
    layers.Input(shape=(img_height, img_width, 3)),
    data_agumentation,
    layers.Lambda(preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(len(classes), activation='softmax')
])

model.summary()

219055592/219055592 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ inception_resnet_v2             │ (None, 5, 5, 1536)     │    54,336,736 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 7)              │        10,759 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 54,347,495 (207.32 MB)

 Trainable params: 10,759 (42.03 KB)

 Non-trainable params: 54,336,736 (207.28 MB)

In [ ]:
classes_file = "./classes.json"
with open(classes_file, "w") as f:
  json.dump(classes, f)

In [7]:


model.compile(
    optimizer='Adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 2823s 2s/step - accuracy: 0.6969 - loss: 0.8614 - val_accuracy: 0.8482 - val_loss: 0.4370
Epoch 2/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 72s 259ms/step - accuracy: 0.8121 - loss: 0.5550 - val_accuracy: 0.8719 - val_loss: 0.3697
Epoch 3/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 72s 258ms/step - accuracy: 0.8324 - loss: 0.4922 - val_accuracy: 0.8830 - val_loss: 0.3457
Epoch 4/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 72s 257ms/step - accuracy: 0.8458 - loss: 0.4535 - val_accuracy: 0.8902 - val_loss: 0.3159
Epoch 5/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 72s 257ms/step - accuracy: 0.8465 - loss: 0.4443 - val_accuracy: 0.8728 - val_loss: 0.3484
Epoch 6/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 73s 260ms/step - accuracy: 0.8550 - loss: 0.4219 - val_accuracy: 0.8929 - val_loss: 0.3040
Epoch 7/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 72s 257ms/step - accuracy: 0.8643 - loss: 0.4026 - val_accuracy: 0.8933 - val_loss: 0.3059
Epoch 8/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 72s 257ms/step - accuracy: 0.8625 - 

In [10]:
model_2 = tf.keras.Sequential([
    layers.Input(shape=(img_height, img_width, 3)),
    data_agumentation,
    layers.Lambda(preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(classes), activation='softmax')
])

model_2.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_3 (Lambda)               │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ inception_resnet_v2             │ (None, 5, 5, 1536)     │    54,336,736 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 256)            │       393,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 54,764,007 (208.91 MB)

 Trainable params: 427,271 (1.63 MB)

 Non-trainable params: 54,336,736 (207.28 MB)

In [11]:
model_2.compile(
    optimizer='Adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

model_2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 104s 297ms/step - accuracy: 0.6975 - loss: 0.8508 - val_accuracy: 0.8571 - val_loss: 0.4220
Epoch 2/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 73s 261ms/step - accuracy: 0.8009 - loss: 0.5910 - val_accuracy: 0.8634 - val_loss: 0.3886
Epoch 3/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 73s 261ms/step - accuracy: 0.8258 - loss: 0.4980 - val_accuracy: 0.8665 - val_loss: 0.3701
Epoch 4/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 72s 258ms/step - accuracy: 0.8442 - loss: 0.4601 - val_accuracy: 0.8763 - val_loss: 0.3249
Epoch 5/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 72s 256ms/step - accuracy: 0.8500 - loss: 0.4329 - val_accuracy: 0.8982 - val_loss: 0.2898
Epoch 6/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 72s 257ms/step - accuracy: 0.8564 - loss: 0.4180 - val_accuracy: 0.8915 - val_loss: 0.3073
Epoch 7/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 72s 258ms/step - accuracy: 0.8690 - loss: 0.3903 - val_accuracy: 0.8942 - val_loss: 0.3059
Epoch 8/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 72s 257ms/step - accuracy: 0.8714 

In [17]:
model_2.save("transfer_learning_accuracy_92.keras")

In [7]:
model_3 = tf.keras.Sequential([
    layers.Input(shape=(img_height, img_width, 3)),
    data_agumentation,
    layers.Lambda(preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(len(classes), activation='softmax')
])

model_3.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ inception_resnet_v2             │ (None, 5, 5, 1536)     │    54,336,736 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       393,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 7)              │           231 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 54,773,671 (208.94 MB)

 Trainable params: 436,935 (1.67 MB)

 Non-trainable params: 54,336,736 (207.28 MB)

In [10]:
model_3.compile(
    optimizer='Adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

model_3.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 3090s 2s/step - accuracy: 0.6419 - loss: 1.0050 - val_accuracy: 0.8482 - val_loss: 0.4480
Epoch 2/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 73s 260ms/step - accuracy: 0.8103 - loss: 0.5914 - val_accuracy: 0.8665 - val_loss: 0.4061
Epoch 3/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 72s 257ms/step - accuracy: 0.8490 - loss: 0.4847 - val_accuracy: 0.8821 - val_loss: 0.3506
Epoch 4/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 72s 256ms/step - accuracy: 0.8618 - loss: 0.4429 - val_accuracy: 0.8973 - val_loss: 0.3180
Epoch 5/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 78s 279ms/step - accuracy: 0.8687 - loss: 0.4176 - val_accuracy: 0.8955 - val_loss: 0.3106
Epoch 6/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 72s 257ms/step - accuracy: 0.8766 - loss: 0.3906 - val_accuracy: 0.8835 - val_loss: 0.3417
Epoch 7/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 72s 257ms/step - accuracy: 0.8874 - loss: 0.3536 - val_accuracy: 0.8955 - val_loss: 0.3022
Epoch 8/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 72s 256ms/step - accuracy: 0.8919 - 

In [11]:
model_3.save("transfer_learning_accuracy_93.keras")